# STEP 5 - Baseline: native JSON-schema compliance of Qwen3-1.7B vs Qwen3-4B
Greedy generation on the 200 gold schemas (100 flat / 100 nested), scored with
the SAME Draft2020-12 validator used everywhere. Picks the model by measured
headroom. Cheapest decisive GPU spend; also proves the eval path on real GPU.

In [ ]:
%pip install -q "transformers>=4.55,<5" accelerate "jsonschema[format]" datasets

In [ ]:
import os, json, re, torch
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")  # private kernel; rotate later
from jsonschema import Draft202012Validator
FMT = Draft202012Validator.FORMAT_CHECKER

# ---- inline copies of the canonical jsonc functions (kernel self-contained) ----
SYSTEM = ("You are a precise JSON generator. Given a JSON Schema, output ONE JSON value "
  "that strictly validates against it. Output ONLY the JSON value - no prose, no "
  "markdown fences, no comments. Include every required property, use only the "
  "properties and types the schema allows, and respect enums, formats, and nesting.")
USER_T = ("Generate a JSON value that validates against this JSON Schema:\n\n{schema}\n\n"
  "Return only the JSON value.")
def render(schema):
    if isinstance(schema, str): schema = json.loads(schema)
    s = json.dumps(schema, sort_keys=True, ensure_ascii=False)
    return [{"role":"system","content":SYSTEM},{"role":"user","content":USER_T.format(schema=s)}]
_FENCE = re.compile(r"^```(?:json)?\s*|\s*```$", re.M)
def extract_json(text):
    t = _FENCE.sub("", text).strip()
    try: return json.loads(t), True
    except Exception: pass
    start=None
    for i,c in enumerate(t):
        if c in "{[": start=i; break
    if start is None: return None, False
    op=t[start]; cl="}" if op=="{" else "]"; d=0; ins=False; esc=False
    for j in range(start,len(t)):
        c=t[j]
        if ins:
            if esc: esc=False
            elif c=="\\": esc=True
            elif c=='"': ins=False
        else:
            if c=='"': ins=True
            elif c==op: d+=1
            elif c==cl:
                d-=1
                if d==0:
                    try: return json.loads(t[start:j+1]), True
                    except Exception: return None, False
    return None, False
def valid(inst, schema):
    if isinstance(schema,str): schema=json.loads(schema)
    try:
        v=Draft202012Validator(schema, format_checker=FMT)
        return len(list(v.iter_errors(inst)))==0
    except Exception: return False

In [ ]:
from datasets import load_dataset
gold = list(load_dataset("ujwal00/jsonschema-gold", split="train", token=os.environ["HF_TOKEN"]))
print("gold:", len(gold), "| flat:", sum(g['complexity_class']=='flat' for g in gold),
      "nested:", sum(g['complexity_class']=='nested' for g in gold))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def score_model(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
    model.eval()
    def gen(prompts):
        out=[]
        for i in range(0,len(prompts),16):
            chunk=prompts[i:i+16]
            enc=tok(chunk,return_tensors="pt",padding=True,truncation=True,max_length=3072).to(model.device)
            with torch.no_grad():
                g=model.generate(**enc,max_new_tokens=640,do_sample=False,pad_token_id=tok.eos_token_id)
            out+=tok.batch_decode(g[:,enc.input_ids.shape[1]:],skip_special_tokens=True)
        return out
    prompts=[]
    for gg in gold:
        try: prompts.append(tok.apply_chat_template(render(gg['schema']),tokenize=False,add_generation_prompt=True,enable_thinking=False))
        except TypeError: prompts.append(tok.apply_chat_template(render(gg['schema']),tokenize=False,add_generation_prompt=True))
    texts=gen(prompts)
    fo=fn=no=nn=pf=0
    for gg,t in zip(gold,texts):
        nested=gg['complexity_class']=='nested'
        if nested: nn+=1
        else: fn+=1
        obj,ok=extract_json(t)
        if not ok: pf+=1; continue
        if valid(obj,gg['schema']):
            if nested: no+=1
            else: fo+=1
    r={"model":model_id,"flat":round(fo/max(fn,1),4),"nested":round(no/max(nn,1),4),
       "all":round((fo+no)/len(gold),4),"parse_fail":round(pf/len(gold),4)}
    print("RESULT",json.dumps(r))
    del model; torch.cuda.empty_cache()
    return r

results=[]
for mid in ["Qwen/Qwen3-1.7B","Qwen/Qwen3-4B-Instruct-2507"]:
    try: results.append(score_model(mid))
    except Exception as e: print("FAILED",mid,e)

In [ ]:
import os,json
os.makedirs("/kaggle/working",exist_ok=True)
json.dump(results,open("/kaggle/working/baseline.json","w"),indent=2)
print(json.dumps(results,indent=2))
print("\nHEADROOM (lower nested = more to gain from fine-tuning):")
for r in results: print(f"  {r['model']}: nested {r['nested']:.0%}, flat {r['flat']:.0%}")